# Canonical decomposition — ENV-map vs SH3 lighting

Compares `scripts/canonical_decomp_env_batch.py` (per-image **environment-map** lighting,
`shader="ct_env"`, results in `results/canonical_env`) against the SH3 batch
(`scripts/canonical_decomp_batch.py`, `shader="ct_sh"`, results in `results/canonical_ablation`),
holding the config fixed so the ONLY difference is the lighting model.

- **Section 2** — overview: SH vs ENV metrics across datasets (albedo / roughness / metallic /
  recon / held-out relighting RMSE).
- **Sections 3–5** — one scene: GT vs SH-est vs ENV-est intrinsics, and the estimated lighting
  (SH `sh_env_map_*` vs ENV `env_map_*`).
- **Section 6** — relight video from the ENV decomposition only (est-vs-GT sweeps live in the
  other notebook).

In [ ]:
import sys, os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

p = Path.cwd()
while not (p / "idr").is_dir() and p != p.parent:
    p = p.parent
os.chdir(p); sys.path[:0] = [str(p), str(p / "scripts")]
print("repo:", p)

from PIL import Image
from canonical_relight_render import (Relighter, load_est_intrinsics, load_gt_intrinsics,
    render_relight_video, _write_mp4, _tonemap_fn, _upscale, _label)
import canonical_decomp_env_batch as ENV

ENV_STUDY = p / "results" / "canonical_env"          # shader = ct_env
SH_STUDY  = p / "results" / "canonical_ablation"     # shader = ct_sh
DEVICE = "cuda"            # set "cpu" if the GPU is busy
print("env study:", ENV_STUDY, "| exists:", ENV_STUDY.exists())
print("sh  study:", SH_STUDY,  "| exists:", SH_STUDY.exists())

## 1. Load both studies

In [ ]:
def load_study(root, lighting):
    rows = []
    for rp in sorted(root.glob("*/*/*/results.json")):     # <dataset>/<scene>/<config>
        dataset, scene, config = rp.parts[-4:-1]
        r = json.loads(rp.read_text()); m = r.get("metrics", {})
        rows.append(dict(lighting=lighting, dataset=dataset, scene=scene, config=config,
                         ds=int(r.get("cfg", {}).get("downsample", 1) or 1),
                         albedo_rmse=m.get("albedo_rmse"), roughness_rmse=m.get("roughness_rmse"),
                         metallic_rmse=m.get("metallic_rmse"), recon_rmse=m.get("recon_rmse"),
                         relighting_rmse=m.get("relighting_rmse"),
                         out_dir=r.get("out_dir", str(rp.parent)), scene_dir=r.get("scene_dir")))
    return rows

df = pd.DataFrame(load_study(ENV_STUDY, "env") + load_study(SH_STUDY, "sh"))
assert len(df), "no results found — run the env and/or SH batch first"
METRICS = ["albedo_rmse", "roughness_rmse", "metallic_rmse", "recon_rmse", "relighting_rmse"]
print("env configs:", sorted(df[df.lighting=='env'].config.unique()))
print("sh  configs:", sorted(df[df.lighting=='sh'].config.unique()))
shared = sorted(set(df[df.lighting=='env'].config) & set(df[df.lighting=='sh'].config))
print("configs in BOTH:", shared)

## 2. Overview — SH vs ENV (matched config)

`CONFIG` is held fixed across both lighting models, so bars compare **ct_sh vs ct_env** directly.
Defaults to a config present in both studies.

In [ ]:
CONFIG = "metallic_l1_1e-2_tv1e-3" if "metallic_l1_1e-2_tv1e-3" in shared else (shared[0] if shared else None)
assert CONFIG, "no config present in both studies — set CONFIG to one from the lists above"
sub = df[df.config == CONFIG]
# clean side-by-side: only datasets that have BOTH lighting models for this config
_have = sub.groupby("dataset")["lighting"].agg(set)
both_ds = sorted(d for d, s in _have.items() if {"sh", "env"} <= s)
datasets = both_ds or sorted(sub["dataset"].unique())   # fall back if none overlap yet
sub = sub[sub.dataset.isin(datasets)]
print("comparing config:", CONFIG, "| datasets with both sh+env:", both_ds)
display(sub.groupby(["dataset", "lighting"])[METRICS].mean().round(4))
lights = ["sh", "env"]
fig, axes = plt.subplots(1, len(METRICS), figsize=(3.5 * len(METRICS), 3.8))
axes = np.atleast_1d(axes); x = np.arange(len(datasets)); w = 0.36
for ax, met in zip(axes, METRICS):
    g = sub.groupby(["dataset", "lighting"])[met].mean()
    for j, lt in enumerate(lights):
        vals = [g.get((d, lt), np.nan) for d in datasets]
        b = ax.bar(x + (j - 0.5) * w, vals, w, label=lt)
        ax.bar_label(b, fmt="%.3f", fontsize=7, padding=1)
    ax.set_xticks(x); ax.set_xticklabels(datasets, fontsize=8); ax.set_title(met, fontsize=10)
    ax.grid(axis="y", ls=":", alpha=0.5)
axes[0].legend(fontsize=9, title="lighting")
fig.suptitle(f"SH3 vs ENV-map lighting — mean RMSE  (config={CONFIG})", fontsize=11)
plt.tight_layout(); plt.show()

## 3. Pick a scene (present in both studies)

In [ ]:
both = (df[(df.config == CONFIG) & (df.lighting == "env")].merge(
        df[(df.config == CONFIG) & (df.lighting == "sh")], on=["dataset", "scene"],
        suffixes=("_env", "_sh")))
assert len(both), f"no scene has BOTH env and sh results for config {CONFIG}"
DATASET = sorted(both["dataset"].unique())[0]
SCENE   = sorted(both[both.dataset == DATASET]["scene"].unique())[0]

row_env = df[(df.lighting=="env") & (df.config==CONFIG) & (df.dataset==DATASET) & (df.scene==SCENE)].iloc[0]
row_sh  = df[(df.lighting=="sh")  & (df.config==CONFIG) & (df.dataset==DATASET) & (df.scene==SCENE)].iloc[0]
ENV_RUN = Path(row_env["out_dir"]); SH_RUN = Path(row_sh["out_dir"])
SCENE_DIR = Path(row_env["scene_dir"]); DS = int(row_env["ds"])
print(f"{DATASET}/{SCENE}  config={CONFIG}  ds={DS}")
print("env run:", ENV_RUN); print("sh  run:", SH_RUN)
display(pd.DataFrame([row_sh[["lighting"]+METRICS], row_env[["lighting"]+METRICS]]).round(4))

## 4. Intrinsics — GT vs SH-est vs ENV-est

In [ ]:
gt  = load_gt_intrinsics(SCENE_DIR, DS); mask = gt["mask"]
sh  = load_est_intrinsics(SH_RUN)
env = load_est_intrinsics(ENV_RUN)
maps = ["albedo", "roughness", "metallic"]
rows = [("GT", gt), ("SH est", sh), ("ENV est", env)]
fig, ax = plt.subplots(len(rows), len(maps), figsize=(3.4 * len(maps), 3.0 * len(rows)))
for i, (name, src) in enumerate(rows):
    for j, k in enumerate(maps):
        m = src[k]; m = m if m.ndim == 3 else m[..., None].repeat(3, -1)
        disp = np.clip(m, 0, 1).copy(); disp[~mask] = 1.0
        if k == "albedo": ax[i, j].imshow(disp)
        else: ax[i, j].imshow(disp[..., 0], cmap="viridis", vmin=0, vmax=1)
        ax[i, j].axis("off")
        if i == 0: ax[i, j].set_title(k, fontsize=10)
    ax[i, 0].set_ylabel(name, fontsize=11, rotation=0, ha="right", va="center")
fig.suptitle(f"{DATASET}/{SCENE} — intrinsics (config={CONFIG})", fontsize=11)
plt.tight_layout(); plt.show()

## 5. Estimated lighting — SH `sh_env_map_*` vs ENV `env_map_*`

The SH batch renders its per-image SH lighting as an equirect env map (`sh_env_map_light_*.png`);
the env batch estimates an env map directly (`env_map_light_*.png`). Shown for the first few lights.

In [ ]:
n_show = 5
sh_maps  = sorted(SH_RUN.glob("sh_env_map_*.png"))[:n_show]
env_maps = sorted(ENV_RUN.glob("env_map_*.png"))[:n_show]
n = min(len(sh_maps), len(env_maps))
if n:
    fig, ax = plt.subplots(2, n, figsize=(2.6 * n, 3.4))
    ax = np.atleast_2d(ax)
    for j in range(n):
        ax[0, j].imshow(np.asarray(Image.open(sh_maps[j])));  ax[0, j].axis("off")
        ax[1, j].imshow(np.asarray(Image.open(env_maps[j]))); ax[1, j].axis("off")
        ax[0, j].set_title(sh_maps[j].stem.replace("sh_env_map_", ""), fontsize=7)
    ax[0, 0].set_ylabel("SH",  fontsize=11, rotation=0, ha="right", va="center")
    ax[1, 0].set_ylabel("ENV", fontsize=11, rotation=0, ha="right", va="center")
    fig.suptitle(f"{DATASET}/{SCENE} — estimated lighting (equirect env maps)", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("missing lighting maps — SH:", len(sh_maps), "ENV:", len(env_maps))

## 6. Relight video — GT vs SH vs ENV

Relights the **GT**, **SH-est**, and **ENV-est** intrinsics through identical geometry under
the same synthetic azimuth sweep (shared tonemap), so the three lighting models sit side by
side. GT is the reference; SH and ENV are the two decompositions.

In [ ]:
import torch
def three_way_relight_video(scene_dir, sh_run, env_run, ds, mode="env_sharp", az_from=-45, az_to=45,
                            n_frames=41, device=DEVICE, fps=18, el=45.0, out_stem=None, panel_px=256):
    gt = load_gt_intrinsics(scene_dir, ds)
    intr = {"GT": gt, "SH": load_est_intrinsics(sh_run), "ENV": load_est_intrinsics(env_run)}
    rel = Relighter(gt["normals"], gt["mask"], device, torch.float32, True)
    kw = dict(sigma_deg=4.0) if mode == "env_sharp" else {}
    az = np.linspace(az_from, az_to, n_frames)
    seq = {k: [rel.render(v, mode, a, el, **kw) for a in az] for k, v in intr.items()}
    tm = _tonemap_fn([f for fr in seq.values() for f in fr])   # one tonemap across all panels
    frames = []
    for i, a in enumerate(az):
        panels = [_label(_upscale(tm(seq[k][i]), panel_px), f"{k} az={a:+.0f}") for k in ("GT", "SH", "ENV")]
        frames.append(np.concatenate(panels, axis=1))
    stem = out_stem or str(Path(env_run) / f"relight_gt_sh_env_{mode}")
    return _write_mp4(frames, stem, fps=fps, ping_pong=True)

from IPython.display import Video, Image as IPyImage
vid = three_way_relight_video(SCENE_DIR, SH_RUN, ENV_RUN, DS)
print("wrote", vid)
display(Video(vid, embed=True, width=900) if vid.endswith(".mp4") else IPyImage(vid))